# Word Embeddings: Distributed Representations, Word2Vec & GloVe

This interactive lab covers:
1. **Distributed Semantics**: Dense continuous vector spaces vs. sparse orthogonal one-hot vectors.
2. **Word2Vec (Skip-Gram with Negative Sampling)**: Optimization via binary logistic regression.
3. **GloVe Matrix Factorization**: Global word co-occurrence count modeling.
4. **Vector Analogy Arithmetic**: Demonstrating semantic compositionality ($v_{\text{king}} - v_{\text{man}} + v_{\text{woman}} \approx v_{\text{queen}}$).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import numpy as np
import torch
import torch.optim as optim
from embedding_engine import (
    SkipGramNegativeSampling, CBOW, GloVeLoss,
    find_most_similar, solve_analogy
)

print('Word Embeddings engine initialized successfully!')

## 1. Word2Vec: Skip-Gram with Negative Sampling (SGNS) Training

We train an SGNS model on co-occurring word index pairs to bring semantically associated tokens closer in embedding space.

In [ ]:
vocab_size = 20
embed_dim = 8
sgns = SkipGramNegativeSampling(vocab_size=vocab_size, embed_dim=embed_dim)
optimizer = optim.Adam(sgns.parameters(), lr=0.05)

# Toy positive context pairs: (0, 1), (0, 2), (10, 11), (10, 12)
center_words = torch.tensor([0, 0, 10, 10])
context_words = torch.tensor([1, 2, 11, 12])
neg_words = torch.randint(3, 10, (4, 4))  # Negative noise tokens

initial_loss = sgns(center_words, context_words, neg_words).item()
for epoch in range(60):
    optimizer.zero_grad()
    loss = sgns(center_words, context_words, neg_words)
    loss.backward()
    optimizer.step()

final_loss = loss.item()
print(f'SGNS Initial Loss: {initial_loss:.4f} -> Final Loss: {final_loss:.4f}')
assert final_loss < initial_loss, 'SGNS loss did not decrease!'

## 2. GloVe Co-Occurrence Loss Function

We evaluate the GloVe weighted least squares objective on word vectors and co-occurrence counts.

In [ ]:
glove = GloVeLoss(x_max=100.0, alpha=0.75)
w_i = torch.randn(4, 8)
w_j = torch.randn(4, 8)
b_i = torch.zeros(4, 1)
b_j = torch.zeros(4, 1)
cooccur = torch.tensor([[5.0], [25.0], [80.0], [150.0]])

g_loss = glove(w_i, w_j, b_i, b_j, cooccur)
print(f'GloVe Weighted Least Squares Loss on Sample Batch: {g_loss.item():.4f}')

## 3. Semantic Vector Analogy Arithmetic

We demonstrate linear relational properties encoded in word vector spaces: $v_{\text{king}} - v_{\text{man}} + v_{\text{woman}} = v_{\text{queen}}$.

In [ ]:
# Synthetic semantic 3D embedding matrix
# Dimensions: [Royalty, Masculinity, Femininity]
embed_matrix = np.array([
    [0.9,  0.8, -0.1],  # 0: king
    [0.0,  0.9, -0.1],  # 1: man
    [0.0, -0.1,  0.9],  # 2: woman
    [0.9, -0.1,  0.9],  # 3: queen
    [0.0,  0.0,  0.0]   # 4: apple
])

idx2word = {0: 'king', 1: 'man', 2: 'woman', 3: 'queen', 4: 'apple'}
word2idx = {w: i for i, w in idx2word.items()}

analogy_target = solve_analogy('man', 'king', 'woman', word2idx, idx2word, embed_matrix, top_k=2)
print('Analogy: "man is to king as woman is to ..."')
for word, score in analogy_target:
    print(f'  Candidate: "{word}" (Cosine Similarity: {score:.4f})')
assert analogy_target[0][0] == 'queen', 'Analogy test failed!'